# Workflow with Pytorch

Mostly using the nn module, that contains all building blocks for neural networks

https://docs.pytorch.org/docs/stable/nn.html

In [ ]:
import torch
import matplotlib.pyplot as plt
print(torch.__version__)

: 

## 1. Prepare Data

The most basic way to get trainable data, is using a linear regression formula to make a straight line with known parameters.

In [ ]:
# y = a + bx
# b
weight = 0.7
# a
bias = 0.3
# x_min
start = 0
# x_max
end = 1
step = 0.01

x = torch.arange(start, end , step)
# to make it a column vector
x = x.unsqueeze(dim=1)
print("X =", x[:10])
y = bias + weight*x
print("Y =", y[:10])

## Creating a Training/Validation/Test Split

While building ML models, it is important to reserve a part of the dat to fit the model (the training set), another part to see how well it is performing on the training (validation) and make decisions on model archtecture; and finally, after a model is finished and trained, a test set to communicate to the world it's performance

In [ ]:
train_size = (int(0.8*len(x)))
val_test_split = (len(x) - train_size)//2 + train_size

x_train_set = x[:train_size]
y_train_set = y[:train_size]
print("X Train Set:", len(x_train_set))
print("Y Train Set:", len(y_train_set))

x_val_set = x[train_size:val_test_split]
y_val_set = y[train_size:val_test_split]
print("X Val Set:", len(x_val_set))
print("Y Val Set:", len(y_val_set))

x_test_set = x[val_test_split:]
y_test_set = y[val_test_split:]
print("X Test Set:", len(x_test_set))
print("Y Test Set:", len(y_test_set))

## Visualizing Data

Very important to make decisions of the best model parameters and archtecture

In [ ]:
# @title
## Create a function to plot predictions
import matplotlib.pyplot as plt

def plot_predictions(train_data,
                     train_labels,
                     test_data,
                     test_labels,
                     predictions=None):
    """
    Plots the training and test dataset and compares predictions
    """
    # a figure where to draw the results
    plt.figure(figsize=(10,7))

    # scatter is a matplotlib function to create a scatter plot of y vs. x
    # with varying marker size and/or color.

    # The train data in blue
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training Data")
    # The test data in green
    plt.scatter(test_data, test_labels, c="g", s=4, label="Testing Data")

    # Plot predictions if they exist
    if predictions is not None:
        # Scatter the predictions in red
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")

    # show legends
    plt.legend(prop={"size": 14})

In [ ]:
plot_predictions(x_train_set, y_train_set, x_test_set, y_test_set)

## 2. Build/Pick a model

Almost every model in pytorch must inherit from torch.nn
- Contains the neccessar for computational graphs (to which Neral Networks belong, in the Torch.nn.Module)
- https://docs.pytorch.org/docs/stable/nn.html

Process for every model:
- Start with random values (weight and bias)
    - Defined as torch.nn.Parameter
- Iteratively adjust random values to better represent the relation between the given input and output
    - Every torch.nn.Module requires a overwrite of the forward() method

In [ ]:
## A simple model to run  linear regression
import torch.nn as nn

# It must use the nn module, from which almost everything inherits
class MyLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        # encapsulates the tensor on nn.Parameters
        # They have the special property of being automatically
        # added to the Module's parameter list (as if they were built-in)
        self.weights = nn.Parameter(torch.randn(1,
                                                # requires gradient,
                                                # allows to record operations
                                                requires_grad=True,
                                                dtype=torch.float))
        self.bias = nn.Parameter(torch.randn(1,
                                             requires_grad=True,
                                             dtype=torch.float))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward method to define the computation of the model for every call
        x: the input data for the model
        Returns a linear regression model
        """
        return self.weights * x + self.bias

- Through 2 main algorithms
    - Gradient Descent
        - The reason why the parameters require_grad = true
            To kep track of how to update the model
        - It makes pytorch track the gradients for this parameter with torch.autograd, besides on the gradient descent.
    - Backpropagation
        - Using torch.optim

The inference mode (also the no_grad mode) in pytoch doesn't adjust the parameters, and turns off the gradient tracking
- Therefore, it's faster, and quite useful to verify the model current state

In [ ]:
# create a manual random seed
torch.manual_seed(42)
# create an instance of the model
model_0 = MyLinearRegression()

print(list(model_0.parameters()))
print(model_0.state_dict())

print(y_test_set[:10])
print("---")

## Predicitions using torch.inference_mode (preferable) or torch.no_grad
with torch.no_grad():
    y_preds = model_0(x_test_set)
    print(y_preds[:10])
with torch.inference_mode():
    y_preds = model_0(x_test_set)
    print(y_preds[:10])

plot_predictions(x_train_set, y_train_set, x_test_set, y_preds)

### 2.1 Pick a loss/cost function / criterion

The function that will infer the overall model error compared to the ideal
- https://docs.pytorch.org/docs/stable/nn.html#loss-functions
- Also called cost function or criterion
- Therefore, the lower the better

Most basic ones:
- nn.L1Loss:
    - Mean Absolute Error
- nn.MSELoss:
    - Mean Squared Error, also called L2 Loss

In [ ]:
loss_fn = nn.L1Loss() # equals to torch.mean(torch.abs(y_pred - y_test))

### 2.2 Pick a Optimizer
---

Takes into account the current output of the loss function and adjust the model
parameters to reduce it.

  - https://docs.pytorch.org/docs/stable/optim.html#torch.optim.Optimizer
  - For simple models, the weight and bias parameters

Requires:

 - Training Loop
    - The parameters are adjusted by the optmier based on the loss functio
 - Testing Loop
    - Only the loss function is applied to verify how well the model currently performs

Most Popular:
 - SGD (Stochastic Gradient Descent): Requires the model parameter and a learning rate (defaults to 0.1, it's the step size for increasing or decreasing the parameters), optionally a momentum () parameter too

In [ ]:
optimizer = torch.optim.SGD(params=model_0.parameters(),
                            lr=0.01)

### 2.3 Build a Training and Testing Loop

0. Loop through the data
    - Each loop is an **epoch**
    - Must set the model mode
        - `model.train()` sets all require_grad parameters to True
            - `model.eval()` turns off settings unrequired to evaluation
        - `with model.no_grad():` or `with model.inference_mode():` (better) turns off gradient tracking and more (but only inside the scope)
1. Forward pass/propagation
    - data being sent and returning from the model's `forward()` functions (possibly more than one) to make predictions on the data
2. Calculate loss
    - compare the the predictions from forward pass with the "ground truth labels", the training/test answers
3. Optmizer zero grad
    - Required to reset the optimizers gradient
4. Loss Backward
    - **Backpropagation**
    - move backwards through the network to calculate the gradients of each parameter (based on the loss), that is, computes the gradient of every parameter with require_grad=True
    - Gradient means the change between 2 points of a function on a cartesian plan (the derivative of a function with more than 1 input variable), therefore going to where the gradient is smaller also reduces the loss function
5. Optimizer Step
    - **Gradient Descent**
    - Use the optimizer to adjust the model's parameters in a way to try improving the loss


## 3. Fit the model to the Data and Make predictions

In [ ]:
epochs = 200

epoch_count = []
loss_values = []
test_loss_values = []

# Training Loop
for epoch in range(epochs):
    # Set the model to training mode
    # That is, all parameters that require gradients to req_gradient=True
    model_0.train()
    # Forward Pass
    y_pred = model_0(x_train_set)
    # Calculate the Loss
    loss = loss_fn(y_pred, y_train_set)
    # Optimizer Zero Grad
    optimizer.zero_grad()
    # Backpropagation
    loss.backward()
    # Optimizer Step / Gradient Descent
    optimizer.step()
    # Testing
    model_0.eval()
    with torch.inference_mode():
        test_pred = model_0(x_test_set)
        test_loss = loss_fn(test_pred, y_test_set)

    if epoch % 10 == 0 or epoch == epochs-1:
        epoch_count.append(epoch)
        loss_values.append(loss.detach().numpy())
        test_loss_values.append(test_loss.detach().numpy())
        print(f"Epoch: {epoch} | Loss: {loss} | Test Loss: {test_loss}")
        print(model_0.state_dict())

## 4. Evaluate the model

In [ ]:
# Test Loop
model_0.eval()
with torch.inference_mode():
    plot_predictions(x_train_set, y_train_set, x_test_set, y_test_set, model_0(x_test_set))

In [ ]:
plt.plot(epoch_count, loss_values, label="Training Loss")
plt.plot(epoch_count, test_loss_values, label="Test Loss")
plt.title("Training and Test Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 5. Improve through experimentation

# 6. Save and reload your trained model

https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html

## `torch.save()`

- Saves a pytorch object on python's pickle format
- The recommended extension is .pth (.pt is also valid)

## `torch.load()`

- Loads a saved pytorch object

## `torch.nn.Module.load_state_dict()`

- Loads only the model state dictionary
- Recommended, as it is more efficient

In [ ]:
from pathlib import Path

# Create models directories:
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Path to save models
MODEL_NAME = "01_model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME
print(f"Model save path: {MODEL_SAVE_PATH}")

In [ ]:
torch.save(obj=model_0.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
loaded_model_0 = MyLinearRegression()
loaded_model_0.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

# Testing the loaded model
loaded_model_0.eval()
with torch.inference_mode():
    loaded_model_preds = loaded_model_0(x_test_set)
    print(f"Loaded model Predictions:\n {loaded_model_preds}")
    print(f"Original model Predictions:\n {model_0(x_test_set)}")

# Putting all Together

In [ ]:
import torch
import matplotlib.pyplot as plt
print(torch.__version__)
# Making it device-agnostic
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device", device)
# Making it reproductible
torch.manual_seed(0)

In [ ]:
# Linear regression data
weight = 0.1
bias = 0.2
start, end = 0, 1
step = 0.01
## Features and Labels
# to make it into a columns vector
X = torch.arange(start, end, step).unsqueeze(dim=1)
Y = weight*X + bias
print(X[:10], Y[:10])

In [ ]:
# Split data
train = 0.8
test = 0.1
val = 0.1

train_size = int(train*len(X))
test_size = int(test*len(X))
val_size = int(val*len(X))
print(train_size, test_size, val_size)

X_train, Y_train = X[:train_size], Y[:train_size]
X_val, Y_val = X[train_size:train_size+val_size], Y[train_size:train_size+val_size]
X_test, Y_test = X[train_size+val_size:train_size+val_size+test_size], Y[train_size+test_size:train_size+val_size+test_size]
print(len(X_train), len(Y_train), len(X_val), len(Y_val), len(X_test), len(Y_test))

In [ ]:
def plot_predictions(train_data,
                     train_labels,
                     test_data,
                     test_labels,
                     val_data,
                     val_labels,
                     predictions=None):
    """
    Plots the training and test dataset and compares predictions
    """
    # a figure where to draw the results
    plt.figure(figsize=(10,7))

    # scatter is a matplotlib function to create a scatter plot of y vs. x
    # with varying marker size and/or color.

    # The train data in blue
    plt.scatter(train_data, train_labels, c="k", s=4, label="Training Data")
    # The validation data in yellow
    plt.scatter(val_data, val_labels, c="b", s=4, label="Validation Data")
    # The test data in green
    plt.scatter(test_data, test_labels, c="g", s=4, label="Testing Data")

    # Plot predictions if they exist
    if predictions is not None:
        # Scatter the predictions in red
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")

    # show legends
    plt.legend(prop={"size": 14})


plot_predictions(X_train, Y_train, X_test, Y_test, X_val, Y_val)

In [ ]:
class LinearRegressionModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Instead of creating the parameters directly,
        # Create a nn layer for them
        # nn.Linear automatically applies a Linear Regression
        # in the incoming data
        self.linear_layer = torch.nn.Linear(in_features=1, # inputs
                                            out_features=1, # outputs
                                            bias=True # That's the default
                                            )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layer(x)

model = LinearRegressionModel()
print(next(model.parameters()).device)
model.to(device=device)
print(next(model.parameters()).device)
print(model.state_dict())

In [ ]:
# Loss function
loss_fn = torch.nn.L1Loss()
# Optimizer
optimizer = torch.optim.Adam(params=model.parameters(),
                            lr=0.001)
# epochs
epochs = 200

X_train = X_train.to(device)
Y_train = Y_train.to(device)
X_val = X_val.to(device)
Y_val = Y_val.to(device)
X_test = X_test.to(device)
Y_test = Y_test.to(device)

for epoch in range(epochs):
    model.train()
    Y_pred = model(X_train)
    loss = loss_fn(Y_pred, Y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.inference_mode():
        val_pred = model(X_val)
        val_loss = loss_fn(val_pred, Y_val)

    if epoch % 10 == 0 or epoch == epochs-1:
        print("epoch:", epoch, " loss:", loss.item(), " val_loss:", val_loss.item())
        # print(model.state_dict())
        # plot_predictions(X_train.cpu(), Y_train.cpu(), X_test.cpu(), Y_test.cpu(), X_val.cpu(), Y_val.cpu(), val_pred.cpu())

model.eval()
with torch.inference_mode():
    test_pred = model(X_test)
    test_loss = loss_fn(test_pred, Y_test)
    print("\n-----\ntest_loss:", test_loss.item())
    print(model.state_dict())
    plot_predictions(X_train.cpu(), Y_train.cpu(), X_test.cpu(), Y_test.cpu(), X_val.cpu(), Y_val.cpu(), test_pred.cpu())

In [ ]:
from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "model_v02.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME
print(f"Model save path: {MODEL_SAVE_PATH}")

torch.save(model.state_dict(), MODEL_SAVE_PATH)

In [ ]:
loaded_model = LinearRegressionModel()

loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
print(loaded_model.to(device))
print(next(loaded_model.parameters()).device)
print(loaded_model.state_dict())

loaded_model.eval()
with torch.inference_mode():
    test_pred = loaded_model(X_test)
    test_loss = loss_fn(test_pred, Y_test)
    plot_predictions(X_train.cpu(), Y_train.cpu(), X_test.cpu(), Y_test.cpu(), X_val.cpu(), Y_val.cpu(), test_pred.cpu())